# Empirical Finance - 20999

# Lab 7: Style Analysis and Factor models

In this lab, we implement **Style Analysis**, a performance evaluation framework introduced by William F. Sharpe in a series of papers beginning in 1989. Afterwards, we run a benchmark **Factor model** to assess whether the investment of interest is actually generating a positive alpha or not.

Following Sharpe's original methodology, we apply it to assess the performance of an investment fund, focusing on a central question: why should an investor be willing to pay management fees? Fees that exceed basic administrative or trading costs can only be justified if the manager consistently delivers returns above the "normal" benchmark, returns that cannot be explained by publicly available information. These excess profits are referred to as "abnormal" returns. Sharpe's method provides a systematic way to estimate both normal and abnormal returns.

In practical terms, the procedure can be summarized as follows. Taking the perspective of investors evaluating a fund manager we select a set of broad market indices and construct a constant relative weights strategies that could, in theory, be implemented independently. The "normal" return is then defined as the out-of-sample performance of this benchmark portfolio, whose weights are chosen to replicate the historical returns of the fund as closely as possible using least squares estimation. If the fund's actual returns consistently outperform this replicable benchmark, the manager can be said to add genuine value. Conversely, if the manager's performance fails to exceed the benchmark, the justification for management fees becomes weak.

In [1]:
import pandas as pd

df = pd.read_excel('Lab7_data.xlsx', index_col=0)
df

,Fund,MKT,SMB,HML,S&P US TREASURY BILL INDEX - PRICE INDEX,FTSE US Government Bond Index 1-10 Yr Maturity - Total Return,FTSE US Government Bond Index 10+ year sector - Total Return,Bloomberg Barclays Security Corporate Bond USD - Average price,Bloomberg Barclays U.S. Mortgage Backed Securities USD - Average price,DJ US LARGE CAP VAL TOTAL STOCK MKT - PRICE INDEX,DJ US LARGE CAP :G TOTAL STOCK MKT. - PRICE INDEX,RUSSELL MIDCAP (EOD) - PRICE INDEX,MSCI US SMALL CAP 1750 - PRICE INDEX,FTSE World Government Bond Index - Total Return,MSCI EUROPE U$ - PRICE INDEX,MSCI JAPAN U$ - PRICE INDEX
2015-01-01,-0.0246,-0.0311,-0.0092,-0.0359,0.000171,0.015986,0.084744,0.019052,0.006304,-0.029669,-0.003366,-0.006768,-0.020332,0.020774,0.005649,0.020620
2015-02-01,0.0642,0.0613,0.0032,-0.0186,0.000019,-0.008842,-0.056836,-0.005943,-0.004013,0.034900,0.060374,0.049332,0.057095,-0.007516,0.048816,0.056775
2015-03-01,-0.0093,-0.0112,0.0307,-0.0038,0.000049,0.006881,0.023404,-0.001193,0.004538,-0.028114,-0.022798,-0.011380,0.002874,0.009413,-0.021446,0.006261
2015-04-01,0.0019,0.0059,-0.0309,0.0182,0.000122,-0.004786,-0.057576,-0.015639,-0.004755,0.027869,0.014785,0.003641,-0.010891,-0.011096,0.031508,0.033906
2015-05-01,-0.0131,0.0136,0.0084,-0.0115,0.000065,-0.000289,-0.020983,-0.012860,-0.003671,-0.002295,0.006220,0.005333,0.013241,-0.007648,-0.017527,0.019002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-08-01,0.0741,0.0762,-0.0095,-0.0288,0.000074,-0.001539,-0.038529,-0.009301,0.002264,0.034197,0.094406,0.033723,0.039196,-0.009187,0.021997,0.058365
2020-09-01,-0.0366,-0.0364,-0.0005,-0.0265,0.000107,0.000204,-0.001865,-0.012988,-0.003135,-0.020259,-0.040105,-0.020897,-0.033297,0.006191,-0.028857,0.002652
2020-10-01,-0.0347,-0.0211,0.0454,0.0431,0.000095,-0.003714,-0.030523,-0.004518,-0.003945,-0.006062,-0.038491,0.013855,0.028528,-0.000955,-0.046152,0.000667
2020-11-01,0.0810,0.1246,0.0705,0.0215,0.000098,-0.001184,-0.007119,0.027335,0.001317,0.111258,0.102313,0.118287,0.152186,-0.002435,0.152903,0.106716


The benchmark set used in the Style Analysis spans a broad spectrum of asset classes, designed to capture the main sources of systematic risk and return across global markets.

It includes **short-term money market instruments** such as Treasury bills, representing cash-equivalent holdings with very short maturities. Within fixed income, the analysis considers **government bonds of varying maturities** (both intermediate- and long-term), **investment-grade corporate bonds**, and **mortgage-related securities**, reflecting different segments of the bond market.

On the equity side, it distinguishes among **large-cap value stocks**, **large-cap growth stocks**, and **mid- and small-cap stocks**, covering firms across capitalization tiers and investment styles within the U.S. market.

Finally, the benchmark set incorporates **international assets**, including non-U.S. government bonds and equities from developed markets outside North America, such as European and Asian stocks.

Together, these categories represent a diversified mix of asset classes and investment styles, allowing for a comprehensive evaluation of a fund's performance and its exposure to different segments of global financial markets.

Alongside the monthly values of these indices, we also include the three factors from the seminal **Fama-French three-factor** model, as well as the **monthly (linear) NET returns of the fund** under analysis. The fund's identity has been anonymized, although the reported return data are real.

In [2]:
df.describe()

,Fund,MKT,SMB,HML,S&P US TREASURY BILL INDEX - PRICE INDEX,FTSE US Government Bond Index 1-10 Yr Maturity - Total Return,FTSE US Government Bond Index 10+ year sector - Total Return,Bloomberg Barclays Security Corporate Bond USD - Average price,Bloomberg Barclays U.S. Mortgage Backed Securities USD - Average price,DJ US LARGE CAP VAL TOTAL STOCK MKT - PRICE INDEX,DJ US LARGE CAP :G TOTAL STOCK MKT. - PRICE INDEX,RUSSELL MIDCAP (EOD) - PRICE INDEX,MSCI US SMALL CAP 1750 - PRICE INDEX,FTSE World Government Bond Index - Total Return,MSCI EUROPE U$ - PRICE INDEX,MSCI JAPAN U$ - PRICE INDEX
count,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000
mean,0.005222,0.010104,-0.000628,-0.007674,0.000854,0.002129,0.005080,0.000836,0.000048,0.004211,0.012666,0.006950,0.006685,0.002459,0.001872,0.005437
std,0.048702,0.045769,0.028306,0.031113,0.000802,0.006864,0.033891,0.020029,0.006112,0.050250,0.052909,0.057611,0.067114,0.009403,0.049414,0.042029
min,-0.137000,-0.135200,-0.082400,-0.138800,-0.000216,-0.020099,-0.094400,-0.111109,-0.024506,-0.259301,-0.204512,-0.305305,-0.349651,-0.020624,-0.207540,-0.120048
25%,-0.019975,-0.004600,-0.019750,-0.023400,0.000159,-0.001577,-0.014942,-0.006177,-0.003251,-0.012888,-0.007661,-0.007690,-0.014822,-0.003732,-0.024223,-0.015066
50%,0.009850,0.010200,0.002650,-0.011950,0.000543,0.001449,0.003688,0.000526,0.000119,0.008830,0.014628,0.010467,0.010471,0.002412,0.006252,0.006584
75%,0.030400,0.032350,0.017050,0.005450,0.001544,0.004685,0.018372,0.008517,0.003902,0.028378,0.038827,0.030918,0.037509,0.006675,0.031622,0.032098
max,0.129400,0.136500,0.070700,0.082100,0.002914,0.021550,0.084744,0.063001,0.015455,0.117060,0.157434,0.151844,0.162449,0.025051,0.152903,0.106716


As mentioned earlier, the reported figures represent monthly returns over a six-year period, from the beginning of 2015 to the end of 2020, yielding a total of 72 monthly observations.

To begin with, it is useful to compute the fund's overall compounded net return. Given that we are working with linear returns, the multi-period compounded linear return is calculated as:
$$
R = \prod_{i=1}^{n} \left( 1 + r_i \right) - 1.
$$
Note the error that would result if we were to simply sum the monthly linear returns, as though they were log returns.


In [3]:
import numpy as np

cumulativeReturn = np.prod( 1 + df['Fund'] )-1
cumulativeReturn

np.float64(0.3368472327547998)

In [4]:
np.sum( df['Fund'] )

np.float64(0.37599999999999995)

The figure may appear "large". However, it represents a six-year performance period, during which, for example, the market factor from the Fama-French three-factor model alone performed roughly three times better.

In [5]:
np.prod( 1 + df['MKT'] )-1

np.float64(0.9154711860159404)

In [6]:
np.sum( df['MKT'] )

np.float64(0.7275)

This occurs despite the fact that the fund and the market factor are highly correlated and, as shown in the previous summary statistics, also display very similar variability (as measured by the sample standard deviation). This suggests that, in practice, the fund's net returns largely mirror those of the market, minus a non-negligible, positive, and almost constant component: most likely, fund's fees.

In [7]:
df[['Fund', 'MKT']].corr()

,Fund,MKT
Fund,1.000000,0.887289
MKT,0.887289,1.000000


We now turn to the core of Sharpe's Style Analysis. Since the "abnormal" returns will be computed recursively, it is convenient to store both the fund's returns and those of the benchmark indices as NumPy objects.

In [8]:
y = df['Fund'].to_numpy()
X = df.iloc[:, 4:].to_numpy()

We begin by setting $m < n$, for example $m = \tfrac{n}{2}$, and let this represent the length of the time window over which the fund manager's investment style is inferred from realized returns. The estimated style is then used to generate out-of-sample forecasts, which can be compared with the fund's actual realized returns.

In [9]:
n = len(y)
n

72

In [10]:
m = int(n/2)
m

36

In [11]:
type(m)

int

We now aim to infer the fund manager's investment style by determining the portfolio weights of a constant-relative weight strategy that best replicate the fund's historical returns over the estimation window.

Specifically, we consider the first $m$ observations in the sample and estimate the weights by minimizing the mean squared error. Since the resulting strategy must represent a valid portfolio, we impose the additional constraint that the portfolio weights sum to one.  

Let $\mathbf{r}^{\pi,1:m} \in \mathbb{R}^{m}$ denote the fund’s returns observed over the first $m$ periods, and let $\mathbf{R}^{1:m} \in \mathbb{R}^{m \times K}$ represent the returns of the $K$ benchmark indices over the same periods. We then compute:

$$
\begin{array}{rl}
\hat{\boldsymbol{\beta}}_{OLS}^{1:m}
&= \underset{\boldsymbol{\beta} \in \mathbb{R}^{K}}{\arg\min}
    \left\| \mathbf{r}^{\pi,1:m} - \mathbf{R}^{1:m}\boldsymbol{\beta} \right\|^2 \\[6pt]
\text{s.t.}
& \mathbf{1}_K' \boldsymbol{\beta} = 1.
\end{array}
$$

Note that the underlying regression model,

$$
\mathbf{r}^{\pi,1:m} = \mathbf{R}^{1:m}\boldsymbol{\beta} + \boldsymbol{\varepsilon},
$$

intentionally excludes an intercept term. In this framework, no asset offers a constant return across all periods that would justify including such a term. Consequently, the usual interpretation of the $R^2$ statistic becomes largely uninformative.

Because of the constraint imposed on the elements of $\boldsymbol{\beta}$, the estimator $\hat{\boldsymbol{\beta}}_{OLS}^{1:m}$ cannot, in principle, be obtained using the standard OLS formula. Nevertheless, in practice, although the resulting unconstrained OLS estimates may yield coefficients whose sum deviates substantially from one, the overall conclusions of the analysis are typically unaffected by this restriction (see the BONUS section below). For the sake of simplicity, we therefore disregard the constraint here and compute $\hat{\boldsymbol{\beta}}_{OLS}^{1:m}$ using the standard OLS approach.


In [12]:
import statsmodels.api as sm

reg = sm.OLS(y[:m], X[:m, :] ).fit()
print(reg.summary())

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.842
Model:                            OLS   Adj. R-squared (uncentered):              0.763
Method:                 Least Squares   F-statistic:                              10.68
Date:                Wed, 01 Jul 2026   Prob (F-statistic):                    6.69e-07
Time:                        23:59:45   Log-Likelihood:                          103.61
No. Observations:                  36   AIC:                                     -183.2
Df Residuals:                      24   BIC:                                     -164.2
Df Model:                          12                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

In [13]:
reg.params.sum()

np.float64(-5.2562798834594595)

The sum of the estimated coefficients differs noticeably from one. To formally verify this, we can perform a hypothesis test defined as follows:
$$
H_0: \sum_{i=1}^K \beta_i = 1
\quad \text{vs.} \quad
H_1: \sum_{i=1}^K \beta_i \neq 1.
$$
This test belongs to the family of significance tests for linear combinations of coefficients. In this case, the vector defining the linear combination is  
$$
\mathbf{R} = \mathbf{1}_K' \in \mathbb{R}^{1 \times K},
$$
since the null hypothesis can be expressed equivalently as  
$$
H_0: \mathbf{R}\boldsymbol{\beta} - 1 = 0
\quad \text{vs.} \quad
H_1: \mathbf{R}\boldsymbol{\beta} - 1 \neq 0.
$$
The corresponding test statistic is given by  
$$
\frac{\mathbf{R}\hat{\boldsymbol{\beta}}_{OLS}^{1:m} - 1}
     {\hat{\sigma}_\varepsilon
     \sqrt{\mathbf{R} \left( \mathbf{X}' \mathbf{X} \right)^{-1} \mathbf{R}'}}
     \sim t(n - K),
$$
where $\mathbf{R}\hat{\boldsymbol{\beta}}_{OLS}^{1:m} = \mathbf{1}_K' \hat{\boldsymbol{\beta}}_{OLS}^{1:m}$ corresponds to the sum of all estimated coefficients, and $\mathbf{R} \left( \mathbf{X}' \mathbf{X} \right)^{-1} \mathbf{R}'
= \mathbf{1}_K' \left( \mathbf{X}' \mathbf{X} \right)^{-1} \mathbf{1}_K$ represents the sum of all the elements of the sampling variance-covariance matrix of the OLS estimator.


In [14]:
V_betaOLS = reg.scale*np.linalg.inv( X[:m, :].T @ X[:m, :] )

tstat = (reg.params.sum() -1)/np.sqrt( V_betaOLS.sum() )
tstat

np.float64(-0.8876114281855811)

We now compute the point forecast of the fund's return based on the previous regression, using as out-of-sample regressors the index returns at time $m+1$, denoted by $\mathbf{r}_{m+1} \in \mathbb{R}^{1 \times K}$.

The forecasted return of the fund at time $m+1$ is therefore given by  
$$
\hat{r}_{m+1}^\pi = \mathbf{r}_{m+1}\hat{\boldsymbol{\beta}}_{OLS}^{1:m}.
$$

Here, $\hat{r}_{m+1}^\pi$ represents the "normal" return, that is, the return expected from passively implementing the constant-weight portfolio strategy $\hat{\boldsymbol{\beta}}_{OLS}^{1:m}$ previously inferred as the fund manager's style. Any deviation of the fund's realized return from this benchmark is interpreted as an "abnormal" return, reflecting either the manager's skill or lack thereof.

Formally, the abnormal return is defined as  
$$
u_{m+1} = r_{m+1}^\pi - \hat{r}_{m+1}^\pi,
$$
that is, the difference between the actual return and the normal benchmark return.

In [15]:
normalReturn = X[m, :] @ reg.params
normalReturn

np.float64(0.05669084416001306)

In [16]:
y[m]

np.float64(0.07769999999999999)

In [17]:
abnormalReturn = y[m] -normalReturn
abnormalReturn

np.float64(0.02100915583998693)

Evidently, the sign of a single abnormal return $u_{m+1}$ is of limited relevance. What truly matters is whether, on average, the fund manager generates positive abnormal returns. To evaluate this in a statistically rigorous manner, we require a time series of abnormal returns from which the sample mean can be computed and tested to determine whether the expected abnormal return is significantly greater than zero.  

To obtain such a sequence, we repeat the procedure used to compute $u_{m+1}$, progressively shifting the estimation window forward through time until all available data points have been utilized.

In [18]:
u = np.zeros(n-m)

for i in range(m):
  reg = sm.OLS(y[i:m+i], X[i:m+i, :] ).fit()
  normalReturn = X[m+i, :] @ reg.params
  u[i] = y[m+i] -normalReturn

In [19]:
u

array([ 2.10091558e-02, -1.54034922e-02,  2.53761779e-02, -1.70249544e-02,
       -4.33229387e-02,  3.00829480e-02,  2.46479654e-02,  3.62259291e-02,
       -1.93309022e-02, -2.08564354e-02,  2.94353060e-02, -2.54315068e-02,
        8.52597899e-03,  4.48408227e-03,  1.43640396e-02,  3.05971971e-02,
        2.18800473e-02, -3.31309490e-02,  1.74096735e-02,  4.17533396e-03,
       -1.37568335e-02, -1.66789998e-02,  1.96908969e-02, -1.09463770e-01,
       -5.86243981e-03, -1.73179855e-02,  1.89613072e-01,  5.83962434e-02,
       -4.15179555e-02,  2.04965451e-02,  1.87831238e-02,  8.12337585e-05,
       -3.23743737e-02, -9.18271447e-03,  3.99007690e-02,  1.23255347e-02])

In [20]:
u.mean()

np.float64(0.0057456945140567225)

We now conduct a simple significance test on the sample mean of the series of abnormal returns.  

Formally, let $\mathbf{u}$ denote the sequence of abnormal returns, treated as an i.i.d. sample from the random variable $u$. We then test the hypotheses  
$$
H_0: \text{E}[u] = 0
\quad \text{vs.} \quad
H_1: \text{E}[u] \neq 0,
$$
using the $t$-ratio of the sample mean $\bar{\mathbf{u}}$ as the test statistic. Under $H_0$, this statistic follows a Student-$t$ distribution with $n - 1$ degrees of freedom, where $n$ is the length of the series $\mathbf{u}$.

In [21]:
mean_u = u.mean()
std_u = u.std()
ste_u = std_u/np.sqrt(len(u))
tratio_u = mean_u/ste_u
tratio_u

np.float64(0.790384063556175)

In [22]:
from scipy.stats import t

pval = 2*t(len(u)-1).cdf(-np.abs(tratio_u))
pval

np.float64(0.4346264295089318)

As we can see, at the conventional 95% significance level, we cannot reject $H_0$, that is, the hypothesis that the expected abnormal return equals zero. In this case, the fund's performance can be fully replicated by the benchmark constant-weight portfolio strategy described earlier. Since such a strategy typically requires frequent rebalancing and thus entails considerable transaction costs, it is already expensive to implement. Therefore, in this scenario, the fund manager should be compensated only to the extent of these implementation costs.

We can also assess the fund's performance using the framework of factor models, specifically by examining whether the fund has generated a nonzero alpha. In this context, the alpha represents the portion of the fund's return that cannot be explained by its exposure to systematic risk factors, and thus serves as a measure of the manager's ability to deliver excess performance.

We can also assess the fund's performance using the framework of factor models, specifically by examining whether the fund has generated a nonzero alpha. In this context, the alpha represents the portion of the fund's return that cannot be explained by its exposure to systematic risk factors, and thus serves as a measure of the manager's ability to deliver excess performance.

The selection of the factors included in the model is of crucial importance, as it determines which sources of systematic risk are accounted for in the analysis. Nevertheless, a widely accepted and practical starting point is provided by the classical Fama-French three-factor model, which extends the traditional Capital Asset Pricing Model (CAPM) by incorporating two additional factors beyond the market risk premium.

Formally, the model can be written as
$$
r_t - r_{f,t} = \alpha + \beta_M (r_{M,t} - r_{f,t}) + \beta_{SMB} \, \text{SMB}_t + \beta_{HML} \, \text{HML}_t + \varepsilon_t,
$$
where $r_t$ denotes the fund's return at time $t$, and $r_{f,t}$ is the risk-free rate.

The three explanatory factors are:
  * Market Risk Premium, $(r_{M,t} - r_{f,t})$: captures the excess return of the overall market portfolio over the risk-free rate, representing broad market exposure.
  * SMB (Small Minus Big): measures the size effect, i.e., the return differential between small-cap and large-cap stocks, reflecting the tendency of smaller firms to earn higher average returns.
  * HML (High Minus Low): captures the value effect, i.e., the return difference between high book-to-market (value) and low book-to-market (growth) stocks.

Together, these factors provide a comprehensive framework for explaining cross-sectional differences in expected returns and allow for a more refined evaluation of a fund’s performance beyond the simple market benchmark.

We begin by examining the correlations between the fund's returns (already expressed in excess of the risk-free rate) and the three factors.

In [23]:
df[ ['Fund', 'MKT', 'SMB', 'HML'] ].corr()

,Fund,MKT,SMB,HML
Fund,1.000000,0.887289,0.269604,0.031179
MKT,0.887289,1.000000,0.430179,0.201984
SMB,0.269604,0.430179,1.000000,0.411174
HML,0.031179,0.201984,0.411174,1.000000


The correlation matrix shows that the fund's returns are highly correlated with the market factor (MKT), suggesting that overall market movements explain a substantial portion of the fund's performance. The correlation with the size factor (SMB) is moderate and positive, indicating some exposure to smaller-cap stocks, while the correlation with the value factor (HML) is very weak, implying limited sensitivity to the value-growth dimension.

In [24]:
FF3 = df[ ['MKT', 'SMB', 'HML'] ]

FF3 = sm.add_constant(FF3)

reg_FF = sm.OLS(df['Fund'], FF3).fit()

print(reg_FF.summary())

                            OLS Regression Results                            
Dep. Variable:                   Fund   R-squared:                       0.816
Model:                            OLS   Adj. R-squared:                  0.807
Method:                 Least Squares   F-statistic:                     100.2
Date:                Wed, 01 Jul 2026   Prob (F-statistic):           6.56e-25
Time:                        23:59:45   Log-Likelihood:                 176.79
No. Observations:                  72   AIC:                            -345.6
Df Residuals:                      68   BIC:                            -336.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0066      0.003     -2.450      0.0

The regression results indicate that the fund's return is almost perfectly aligned with the market factor, as shown by the estimated market beta, which is essentially equal to one. This suggests that the fund's performance closely tracks overall market movements, leaving little evidence of distinctive active management.  

The intercept, namely the **alpha**, is negative and statistically significant at the 5\% level, implying that, after controlling for exposure to the standard Fama-French factors, the fund underperforms its expected return.

In [25]:
semipartialR2 = reg_FF.tvalues**2*(1-reg_FF.rsquared)/reg_FF.df_resid
semipartialR2

,0
const,0.016276
MKT,0.735261
SMB,0.005457
HML,0.012888


## BONUS: Implementing the constrained OLS estimator in the Style Analysis

As previously noted, the OLS estimators computed above cannot be directly interpreted as portfolio weights, since they are not constrained to sum to one.  

To enforce this property, one could either implement a numerical optimization routine that explicitly includes the constraint, or, as explained in the Lecture Notes, apply the standard OLS formula to a suitably transformed set of dependent and independent variables. The idea is to eliminate the constraint through an appropriate reparameterization of the control variable.  

Specifically, subtract one column of $\mathbf{R}$, say, the first one, from all other columns and simultaneously from $\mathbf{r}$. Formally, define  
$$
\tilde{\mathbf{R}} = \mathbf{R}_{:,2:k} - \mathbf{R}_{:,1},
\qquad
\tilde{\mathbf{r}} = \mathbf{r} - \mathbf{R}_{:,1},
$$
so that $\tilde{\mathbf{R}} \in \mathbb{R}^{n \times (k-1)}$ and $\tilde{\mathbf{r}} \in \mathbb{R}^n$.  
Let $\tilde{\boldsymbol{\beta}} = \boldsymbol{\beta}_{2:k} \in \mathbb{R}^{k-1}$ denote the subvector of portfolio weights excluding the first one.  

Now consider the OLS estimator of $\tilde{\boldsymbol{\beta}}$ in the regression of $\tilde{\mathbf{r}}$ on $\tilde{\mathbf{R}}$, denoted by $\hat{\tilde{\boldsymbol{\beta}}}_{OLS}$. The $k$-dimensional vector whose first entry is the complement to one of the sum of $\hat{\tilde{\boldsymbol{\beta}}}_{OLS}$ and whose remaining entries are exactly $\hat{\tilde{\boldsymbol{\beta}}}_{OLS}$, namely  
$$
\begin{bmatrix}
1 - \mathbf{1}_{k-1}' \hat{\tilde{\boldsymbol{\beta}}}_{OLS} \\
\hat{\tilde{\boldsymbol{\beta}}}_{OLS}
\end{bmatrix},
$$
is the optimal solution to  
$$
\begin{array}{ll}
\underset{\boldsymbol{\beta} \in \mathbb{R}^{K}}{\arg\min} &
\left\| \mathbf{r}^{\pi,1:m} - \mathbf{R}\boldsymbol{\beta} \right\|^2 \\[6pt]
\text{s.t.} & \mathbf{1}_K' \boldsymbol{\beta} = 1.
\end{array}
$$

In [26]:
ytilde = y - X[:,0]
Xtilde = X[:,1:] - X[:,0].reshape(-1,1)

In [27]:
reg = sm.OLS(ytilde[:m+i], Xtilde[:m+i,:]).fit()

In [28]:
constrainedBeta = np.append(1-reg.params.sum(), reg.params)
constrainedBeta

array([ 1.75164506, -2.96406436,  0.26795869, -0.03582733,  0.27661716,
       -0.12642382,  0.86745009, -0.38025294,  0.21307812,  0.90601893,
       -0.22768739,  0.45148778])

In [29]:
constrainedBeta.sum()

np.float64(1.0)

In [30]:
normalRet = X[m, :] @ constrainedBeta
abnormalRet = y[m] - normalRet
abnormalRet

np.float64(0.004412415789110169)

In [31]:
u_constrained = np.zeros(n-m)

for i in range(m):
  reg = sm.OLS(ytilde[i:m+i], Xtilde[i:m+i,:]).fit()
  constrainedBeta = np.append(1-reg.params.sum(), reg.params)
  normalRet = X[m+i, :] @ constrainedBeta
  u_constrained[i] = y[m+i] - normalRet

In [32]:
u_constrained.mean()

np.float64(0.0032590485208556083)

In [33]:
tratio_uconstrained = u_constrained.mean()/u_constrained.std()*np.sqrt(len(u_constrained))
tratio_uconstrained

np.float64(0.4206369849739682)

In [34]:
pval_constrained = 2*t(len(u_constrained)-1).cdf(-np.abs(tratio_uconstrained))
pval_constrained

np.float64(0.6765917870357182)